# 02 · Recogida de datos de toda la región

Cada respuesta se guarda en `data/raw/` y, si el archivo ya existe, no se vuelve a pedir: ejecutar otra vez solo descarga lo que falta.

1. INE (API): población por municipio y empresas del sector por municipio (competencia).
2. Wikipedia (scraping): festivos municipales.
3. Portal BASE (dados.gov.pt): lista de archivos y descarga de los años elegidos.
4. OpenStreetMap (API Overpass): organizaciones de los 66 municipios.
5. OpenStreetMap por freguesia: solo los municipios del piloto.

OpenStreetMap va al final porque es la parte lenta: si un servidor falla, las otras fuentes ya están guardadas.

> **Cómo leer este notebook.** Antes de cada celda de código hay un bloque **Cómo funciona**: qué hace la celda, por qué se hace así y qué herramientas usa. Cuando algo no se ha visto en clase, se marca como **Nuevo** y se explica con más calma. Las funciones que se llaman (`recoleccion.…`, `cleaning.…`, `analisis.…`) están en la carpeta `src/`, con su explicación en el docstring.

**Cómo funciona**

- **Qué hace:** prepara el notebook. Calcula la carpeta raíz del proyecto, la añade a `sys.path` para poder importar los módulos de `src/`, los recarga y lee `config.yaml` en un diccionario.
- **Por qué así:** el código que se repite vive en `src/` y los valores que pueden cambiar (municipios, URLs, rutas, pesos) en `config.yaml`. Así el notebook solo ordena los pasos. `importlib.reload` vuelve a leer `src/` sin reiniciar el kernel cuando se cambia una función.
- **Herramientas:** `import`, diccionarios, `pathlib.Path` (rutas que funcionan igual en Windows y en Mac).
- **Nuevo (no visto en clase):** `importlib.reload`, `sys.path` y `pathlib`.
- **Además:** junta los municipios de los cuatro distritos en una sola lista y comprueba con `assert` que son 66: si falta o sobra uno en `config.yaml`, el notebook se para aquí con un mensaje claro.

In [45]:
import importlib
import sys
from pathlib import Path

import pandas as pd

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(RAIZ))

from src import recoleccion

# Recarga el código de src/ por si ha cambiado (así no hace falta reiniciar el kernel)
importlib.reload(recoleccion)

config = recoleccion.cargar_config(RAIZ / "config.yaml")
RUTA_RAW = RAIZ / config["rutas"]["raw"]
municipios = [m for lista in config["region"]["distritos"].values() for m in lista]
assert len(municipios) == 66, f"Esperábamos 66 municipios y hay {len(municipios)}"

## 1. INE: población y empresas del sector

**Cómo funciona**

- **Qué hace:** pide (o lee) la población de 2023 de todo el país y cuenta los registros.
- **Por qué así:** mismo patrón de caché que en el 01. Se pide todo el país en una sola petición y se filtra después: es más rápido que una petición por municipio.
- **Herramientas:** f-strings, `.get()` con valor por defecto, bucle sobre `.items()`.

In [46]:
ine = config["ine"]

ruta_pob = RUTA_RAW / f"ine_{ine['indicador_poblacion']}_{ine['periodo']}.json"
if ruta_pob.exists():
    datos_pob = recoleccion.leer_json(ruta_pob)
else:
    datos_pob = recoleccion.pedir_datos_ine(ine["indicador_poblacion"], ine, ine["periodo"])
    recoleccion.guardar_json(datos_pob, ruta_pob)

primero = datos_pob[0] if isinstance(datos_pob, list) else datos_pob
for periodo, registros in primero.get("Dados", {}).items():
    print("Población", periodo, "- registros:", len(registros))

Población 2023 - registros: 19608


**Cómo funciona**

- **Qué hace:** guarda los metadatos del indicador de empresas y pide, una a una, las empresas de cada actividad del sector (bordados, confección, impresión, comercio de ropa y de deporte).
- **Por qué así:** la API del INE devuelve una actividad por petición. `recoger_ine_por_categorias` guarda cada respuesta, hace una pausa de 2 s entre peticiones y, si una falla, sigue con las demás (`try/except`) para reintentarla en la próxima ejecución.
- **Herramientas:** `try/except`, bucles, `time.sleep`.
- **Nuevo (no visto en clase):** los códigos CAE (la clasificación portuguesa de actividades económicas).

In [47]:
# Metadatos del indicador de empresas: dimensiones y códigos CAE disponibles
ruta_meta_emp = RUTA_RAW / f"ine_metadatos_{ine['indicador_empresas']}.json"
if not ruta_meta_emp.exists():
    recoleccion.guardar_json(recoleccion.pedir_metadatos_ine(ine["indicador_empresas"], ine), ruta_meta_emp)

# Competencia: número de empresas por municipio en cada actividad económica (CAE) del sector
empresas_por_cae = recoleccion.recoger_ine_por_categorias(
    ine["indicador_empresas"], list(ine["cae_competencia"]), ine, ine["periodo_empresas"], RUTA_RAW
)

13991: ya estaba guardado
14120: ya estaba guardado
14131: ya estaba guardado
14190: ya estaba guardado
18120: ya estaba guardado
47711: ya estaba guardado
47640: ya estaba guardado


## 2. Wikipedia: festivos municipales

**Cómo funciona**

- **Qué hace:** descarga la página «Feriados em Portugal» de Wikipedia, extrae las filas de las tablas de festivos municipales y las guarda.
- **Por qué así:** es la parte de web scraping. Se identifica el proyecto con un user agent y se guarda el HTML para no volver a pedirlo. `extraer_feriados` busca la sección por su `id` y recorre sus tablas. El problema: cuando varios festivos caen el mismo día, la celda de la fecha ocupa varias filas (`rowspan`) y las filas siguientes solo tienen dos celdas. Por eso la función recuerda la última fecha (`fecha_actual`). Sin eso faltaban 12 municipios.
- **Herramientas:** `requests`, `BeautifulSoup` (`find`, `find_all`, `get_text`), bucles y condiciones.

In [48]:
wiki = config["wikipedia"]
html = recoleccion.descargar_html(wiki["url_feriados"], RUTA_RAW / "wikipedia_feriados.html", wiki["user_agent"])
filas_feriados = recoleccion.extraer_feriados(html)
recoleccion.guardar_json(filas_feriados, RUTA_RAW / "wikipedia_feriados_filas.json")

print("Filas extraídas:", len(filas_feriados))
pd.DataFrame(filas_feriados)[["fecha_texto", "nombre_festivo"]].head()

Filas extraídas: 174


,fecha_texto,nombre_festivo
0,Segunda-feira após a Páscoa (entre 23 de Março...,Segunda-feira de Páscoa
1,Segunda-feira após a Páscoa (entre 23 de Março...,Nossa Senhora da Luz
2,Segunda-feira após a Páscoa (entre 23 de Março...,Nossa Senhora da Boa Viagem
3,Segunda-feira após a Páscoa (entre 23 de Março...,São Gregório
4,Segunda-feira após a Páscoa (entre 23 de Março...,Nossa Senhora da Graça


## 3. Portal BASE: archivos de contratos

**Cómo funciona**

- **Qué hace:** vuelve a listar los archivos del Portal BASE (lee la lista guardada) con su tamaño en MB.
- **Por qué así:** sirve para elegir los nombres exactos de los archivos de cada año (`contratos2020.xlsx`…).
- **Herramientas:** `pd.to_numeric`, división, `round`, `sort_values`.

In [49]:
ruta_info_base = RUTA_RAW / "base_dataset_info.json"
if ruta_info_base.exists():
    info_base = recoleccion.leer_json(ruta_info_base)
else:
    info_base = recoleccion.listar_recursos_dataset(config["base"]["api_dataset"])
    recoleccion.guardar_json(info_base, ruta_info_base)

recursos = pd.DataFrame(info_base["resources"])
recursos["mb"] = (pd.to_numeric(recursos["filesize"], errors="coerce") / 1_000_000).round(1)
recursos[["title", "format", "mb"]].sort_values("title")

,title,format,mb
2,contratos2012.xlsx,xlsx,16.4
29,contratos2012.zip,zip,12.0
15,contratos2013.xlsx,xlsx,20.0
28,contratos2013.zip,zip,14.8
14,contratos2014.xlsx,xlsx,24.8
27,contratos2014.zip,zip,17.5
13,contratos2015.xlsx,xlsx,25.6
26,contratos2015.zip,zip,18.4
12,contratos2016.xlsx,xlsx,27.4
25,contratos2016.zip,zip,20.0


**Cómo funciona**

- **Qué hace:** descarga los Excel de 2020 a 2025 que aún no estén en `data/raw/base/`.
- **Por qué así:** `recursos[recursos["title"] == ...]` filtra la fila del archivo de cada año e `.iloc[0]` toma la primera. `if not ruta.exists()` evita descargar dos veces.
- **Herramientas:** filtro booleano, `iloc`, `descargar_archivo` (descarga por partes).

In [50]:
# Descarga de los contratos de los años elegidos (en Excel). Se activa después de revisar la lista.
DESCARGAR = True

if DESCARGAR:
    for ano in config["base"]["anos"]:
        fila = recursos[recursos["title"] == f"contratos{ano}.xlsx"].iloc[0]
        ruta = RUTA_RAW / "base" / fila["title"]
        if not ruta.exists():
            print("Descargando", fila["title"], fila["mb"], "MB")
            recoleccion.descargar_archivo(fila["url"], ruta)

**Cómo funciona**

- **Qué hace:** lee solo 3 filas del primer Excel para ver sus 39 columnas y guarda esa muestra.
- **Por qué así:** leer un Excel entero tarda minutos; con `nrows=3` se decide qué columnas sirven sin esperar. `.T` gira la tabla para leer las columnas como filas.
- **Herramientas:** `pd.read_excel(nrows=...)`, `.T`, `to_dict(orient="records")`.

In [51]:
# Primer vistazo al archivo de 2023: columnas y tres filas (leer el Excel entero tarda unos minutos)
primer_archivo = RUTA_RAW / "base" / f"contratos{config['base']['anos'][0]}.xlsx"
muestra = pd.read_excel(primer_archivo, nrows=3)
recoleccion.guardar_json(
    {"columnas": muestra.columns.tolist(), "filas": muestra.astype(str).to_dict(orient="records")},
    RUTA_RAW / "base_muestra.json",
)
print(muestra.shape[1], "columnas")
muestra.T

39 columnas


,0,1,2
idcontrato,5642587,5801400,6038721
nAnuncio,1487/2018,497/2019,8200/2019
TipoAnuncio,Aviso de prorrogação de prazo,Anúncio de procedimento,Anúncio de procedimento
idINCM,411803103,411994379,412492163
tipoContrato,Empreitadas de obras públicas,Aquisição de bens móveis,Aquisição de serviços
idprocedimento,3621473,3800257,4124458
tipoprocedimento,Concurso público,Concurso público,Concurso público
objectoContrato,Empreitada de Beneficiação da Estação de Trata...,Aquisição de cavilhas e travessas de madeira d...,Aquisição de serviços com vista ao forneciment...
descContrato,Empreitada de Beneficiação da Estação de Trata...,Aquisição de cavilhas e travessas de madeira d...,Aquisição de serviços com vista ao forneciment...
adjudicante,501262997 - Município de Mangualde,"503933813 - Infraestruturas de Portugal, S. A.",502736208 - Faculdade de Direito da Universida...


**Cómo funciona**

- **Qué hace:** primer filtro de los contratos: de cada Excel anual se queda con la ropa y los artículos promocionales de todo el país (12.350 contratos).
- **Por qué así:** un contrato entra si su código CPV empieza por 18 (ropa) o es 39294100 (artículos promocionales), sin calzado (188), o si su descripción tiene palabras como «t-shirt», «fardamento» o «merchandising». El resultado de cada año se guarda en CSV junto al Excel: la segunda vez tarda segundos en vez de minutos.
- **Herramientas:** `read_excel(usecols=...)`, `str.startswith`, `str.contains` con expresión regular, `|` (o) entre filtros, `to_csv`, `pd.concat`.
- **Nuevo (no visto en clase):** los códigos CPV (el vocabulario común de los contratos públicos) y `normalize("NFKD")` para quitar acentos antes de buscar palabras.

In [52]:
# Primer filtro: solo los contratos de ropa y artículos promocionales (CPV 18 sin calzado, 39294100,
# o palabras como "t-shirt", "fardamento" o "merchandising" en la descripción).
# La primera vez lee los Excel enteros (unos minutos por año) y guarda contratosAAAA_ropa_v2.csv
# junto a cada Excel; las siguientes veces lee directamente esos CSV.
rutas_base = [RUTA_RAW / "base" / f"contratos{ano}.xlsx" for ano in config["base"]["anos"]]
contratos_ropa = recoleccion.leer_contratos_ropa(rutas_base, config["base"]["prefijos_cpv"])

print("Contratos de ropa o artículos promocionales en Portugal:", len(contratos_ropa))
contratos_ropa["Ano"].value_counts().sort_index()

contratos2020.xlsx - leído del archivo filtrado
contratos2021.xlsx - leído del archivo filtrado
contratos2022.xlsx - leído del archivo filtrado
contratos2023.xlsx - leído del archivo filtrado
contratos2024.xlsx - leído del archivo filtrado
contratos2025.xlsx - leído del archivo filtrado
Contratos de ropa o artículos promocionales en Portugal: 12350


Ano
2020    3879
2021    2074
2022    1430
2023    1630
2024    1687
2025    1650
Name: count, dtype: int64

## 4. OpenStreetMap: los 66 municipios

**Cómo funciona**

- **Qué hace:** pide a Overpass las organizaciones de los 66 municipios, uno a uno, y guarda cada respuesta.
- **Por qué así:** una petición por municipio es más ligera y, si una falla, no se pierde lo demás. `pedir_overpass` prueba tres servidores públicos: si uno está saturado (errores 429 o 504) pasa al siguiente y, si fallan todos, espera cada vez más. Al volver a ejecutar solo se piden los que faltan.
- **Herramientas:** bucles, `try/except`, `requests.post`, `time.sleep`.

In [53]:
resultados_osm = recoleccion.recoger_osm_region(
    municipios, config["overpass"], config["overpass"]["etiquetas"], RUTA_RAW
)

 1/66 Arganil: 46 elementos
 2/66 Cantanhede: 53 elementos
 3/66 Coimbra: 172 elementos
 4/66 Condeixa-a-Nova: 21 elementos
 5/66 Figueira da Foz: 67 elementos
 6/66 Góis: 12 elementos
 7/66 Lousã: 17 elementos
 8/66 Mira: 18 elementos
 9/66 Miranda do Corvo: 14 elementos
10/66 Montemor-o-Velho: 27 elementos
11/66 Oliveira do Hospital: 39 elementos
12/66 Pampilhosa da Serra: 24 elementos
13/66 Penacova: 14 elementos
14/66 Penela: 11 elementos
15/66 Soure: 27 elementos
16/66 Tábua: 19 elementos
17/66 Vila Nova de Poiares: 19 elementos
18/66 Armamar: 22 elementos
19/66 Carregal do Sal: 14 elementos
20/66 Castro Daire: 27 elementos
21/66 Cinfães: 50 elementos
22/66 Lamego: 44 elementos
23/66 Mangualde: 18 elementos
24/66 Moimenta da Beira: 21 elementos
25/66 Mortágua: 11 elementos
26/66 Nelas: 28 elementos
27/66 Oliveira de Frades: 10 elementos
28/66 Penalva do Castelo: 6 elementos
29/66 Penedono: 5 elementos
30/66 Resende: 18 elementos
31/66 Santa Comba Dão: 9 elementos
32/66 São João da

**Cómo funciona**

- **Qué hace:** cuenta los elementos de cada municipio (1.824 en total) y busca municipios sin datos.
- **Por qué así:** un municipio con 0 suele indicar que su nombre en `config.yaml` no coincide con el de OpenStreetMap. Es una comprobación antes de limpiar.
- **Herramientas:** dict comprehension, `pd.Series`, filtro booleano, `sort_values`.

In [54]:
elementos_por_municipio = pd.Series({m: len(e) for m, e in resultados_osm.items()}, name="elementos")
print("Total de elementos:", elementos_por_municipio.sum())

# Un municipio con 0 elementos suele indicar que su nombre no coincide con el de OpenStreetMap
print("Municipios sin datos:", elementos_por_municipio[elementos_por_municipio == 0].index.tolist())
elementos_por_municipio.sort_values(ascending=False).head(10)

Total de elementos: 1824
Municipios sin datos: []


Coimbra            172
Viseu              130
Castelo Branco      81
Covilhã             73
Figueira da Foz     67
Fundão              64
Tondela             59
Guarda              59
Cantanhede          53
Cinfães             50
Name: elementos, dtype: int64

## 5. Freguesias de los municipios del piloto

Para el piloto necesitamos saber en qué freguesia está cada organización (por ejemplo, qué junta corresponde a cada club). La consulta por freguesia es más pesada, así que solo se hace para los municipios del piloto.

**Cómo funciona**

- **Qué hace:** repite la recogida para los 6 municipios del piloto, con una consulta por freguesia.
- **Por qué así:** esa consulta devuelve primero cada freguesia y después lo que hay dentro: así sabemos la freguesia de cada organización y qué juntas faltan. Es más pesada, por eso solo se hace en el piloto.
- **Herramientas:** la misma función con `por_freguesia=True`.
- **Nuevo (no visto en clase):** `foreach` y `map_to_area` de Overpass QL.

In [55]:
resultados_freguesias = recoleccion.recoger_osm_region(
    config["region"]["piloto"], config["overpass"], config["overpass"]["etiquetas"], RUTA_RAW,
    por_freguesia=True,
)

 1/6 Oliveira do Hospital: 59 elementos
 2/6 Arganil: 60 elementos
 3/6 Carregal do Sal: 19 elementos
 4/6 Nelas: 35 elementos
 5/6 Seia: 57 elementos
 6/6 Tábua: 30 elementos
